# 01 · Factor Models: CAPM → FF3 → FF5

**Goal.** Take a set of standard test portfolios and explain their returns with three
nested asset-pricing models, adding factors as we go:

1. **CAPM** — one factor: the market (`Mkt-RF`)
2. **Fama-French 3** — adds size (`SMB`) and value (`HML`)
3. **Fama-French 5** — adds profitability (`RMW`) and investment (`CMA`)

**The thing to watch.** Each model produces an *alpha* (intercept) per portfolio —
the average return the factors *fail* to explain. If the added factors are real
sources of priced risk, the alphas should **shrink toward zero** as we move
CAPM → FF3 → FF5. That trend is the whole point of this notebook.

---

### Roadmap

| Step | What | Status |
|------|------|--------|
| **1** | Load factor data + test portfolios, inspect & sanity-check | ✅ this notebook |
| 2 | Build excess returns for the test portfolios | next |
| 3 | Run CAPM on every portfolio | next |
| 4 | Run FF3 | next |
| 5 | Run FF5 | next |
| 6 | Compare alphas across models (table, heatmap, t-stats, GRS) | next |


## Step 0 · Setup

We import the `get_french` helper from `src/data_loader.py`, which downloads a
Ken French dataset and caches it locally as parquet (under `data/`, gitignored).
All French returns are **in percent per month**.

In [1]:
import sys, os
sys.path.append(os.path.abspath(".."))   # so we can import from src/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data_loader import get_french

pd.set_option("display.float_format", lambda x: f"{x:.4f}")
plt.rcParams["figure.figsize"] = (10, 5)

/Users/Parimah/anaconda3/lib/python3.11/site-packages/pandas/core/computation/expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.4' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/Users/Parimah/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


## Step 1 · Load the data

We need two things:

1. **Factors** — we pull the **5-factor** file `F-F_Research_Data_5_Factors_2x3`,
   which contains `Mkt-RF, SMB, HML, RMW, CMA, RF` in one place. CAPM uses just
   `Mkt-RF`; FF3 uses `Mkt-RF, SMB, HML`; FF5 uses all five. Pulling one file keeps
   the risk-free rate `RF` consistent across all three models.

   > ⚠️ *Caveat:* the `SMB` in the 5-factor file is built slightly differently from
   > the `SMB` in the original 3-factor file (it averages across more sorts). For a
   > learning exercise this is the standard, clean choice — just know the FF3 here
   > uses the 5-factor-style `SMB`.

2. **Test portfolios** — the classic `25_Portfolios_5x5`: 25 portfolios formed on
   a 5×5 sort of **size** × **book-to-market**. These are the canonical left-hand-side
   assets for testing Fama-French models.

In [2]:
# Monthly sample. French data starts 1963-07 for these series.
START, END = "1963-07", "2024-12"

factors = get_french("F-F_Research_Data_5_Factors_2x3", start=START, end=END)
portfolios = get_french("25_Portfolios_5x5", start=START, end=END)

print("factors:   ", factors.shape, "->", list(factors.columns))
print("portfolios:", portfolios.shape)

factors:    (738, 6) -> ['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA', 'RF']
portfolios: (738, 25)


### Inspect the factors

What each column means (all are monthly returns, in %):

| Factor | Name | What it captures | Long − short |
|---|---|---|---|
| `Mkt-RF` | Market | Excess return of the whole market over the risk-free rate | Market − T-bill |
| `SMB` | Size | Small-cap stocks tend to beat large-cap | Small − Big |
| `HML` | Value | Cheap (high book/price) beats expensive (growth) | High − Low B/M |
| `RMW` | Profitability | Profitable firms beat unprofitable ones | Robust − Weak |
| `CMA` | Investment | Conservative (low-investment) firms beat aggressive ones | Conservative − Aggressive |
| `RF` | Risk-free | The risk-free rate (T-bill); not a factor — used to form excess returns | — |

In [3]:
factors.head()

,Mkt-RF,SMB,HML,RMW,CMA,RF
Date,,,,,,
1963-07-31 23:59:59.999999,-0.3900,-0.4800,-0.8100,0.6400,-1.1500,0.2700
1963-08-31 23:59:59.999999,5.0800,-0.8000,1.7000,0.4000,-0.3800,0.2500
1963-09-30 23:59:59.999999,-1.5700,-0.4300,0.0000,-0.7800,0.1500,0.2700
1963-10-31 23:59:59.999999,2.5400,-1.3400,-0.0400,2.7900,-2.2500,0.2900
1963-11-30 23:59:59.999999,-0.8600,-0.8500,1.7300,-0.4300,2.2700,0.2700


In [4]:
factors.describe().T[["mean", "std", "min", "max"]]

,mean,std,min,max
Mkt-RF,0.5867,4.4760,-23.1900,16.1000
SMB,0.1930,3.0361,-15.5400,18.4600
HML,0.2819,2.9714,-13.8300,12.8600
RMW,0.2810,2.2172,-18.9500,13.0500
CMA,0.2509,2.0574,-7.0800,9.0100
RF,0.3637,0.2636,0.0000,1.3500


### Inspect the test portfolios

Columns are labeled `SIZE B/M`, e.g. `SMALL LoBM` (small-cap, low book-to-market /
growth) through `BIG HiBM` (large-cap, high book-to-market / value).

The 25 columns come in a fixed **row-major order**: outer loop is **size** (ME1→ME5,
small→big), inner loop is **book-to-market** (BM1→BM5, growth→value). Laid out as the
5×5 grid:

| Size \\ B/M | BM1 (Growth) | BM2 | BM3 | BM4 | BM5 (Value) |
|---|---|---|---|---|---|
| **ME1 (Small)** | `SMALL LoBM` | `ME1 BM2` | `ME1 BM3` | `ME1 BM4` | `SMALL HiBM` |
| **ME2** | `ME2 BM1` | `ME2 BM2` | `ME2 BM3` | `ME2 BM4` | `ME2 BM5` |
| **ME3** | `ME3 BM1` | `ME3 BM2` | `ME3 BM3` | `ME3 BM4` | `ME3 BM5` |
| **ME4** | `ME4 BM1` | `ME4 BM2` | `ME4 BM3` | `ME4 BM4` | `ME4 BM5` |
| **ME5 (Big)** | `BIG LoBM` | `ME5 BM2` | `ME5 BM3` | `ME5 BM4` | `BIG HiBM` |

The four corners are the extremes: `SMALL LoBM` (small growth), `SMALL HiBM`
(small value), `BIG LoBM` (large growth), `BIG HiBM` (large value).

In [5]:
print(list(portfolios.columns))
portfolios.head()

['SMALL LoBM', 'ME1 BM2', 'ME1 BM3', 'ME1 BM4', 'SMALL HiBM', 'ME2 BM1', 'ME2 BM2', 'ME2 BM3', 'ME2 BM4', 'ME2 BM5', 'ME3 BM1', 'ME3 BM2', 'ME3 BM3', 'ME3 BM4', 'ME3 BM5', 'ME4 BM1', 'ME4 BM2', 'ME4 BM3', 'ME4 BM4', 'ME4 BM5', 'BIG LoBM', 'ME5 BM2', 'ME5 BM3', 'ME5 BM4', 'BIG HiBM']


,SMALL LoBM,ME1 BM2,ME1 BM3,ME1 BM4,SMALL HiBM,ME2 BM1,ME2 BM2,ME2 BM3,ME2 BM4,ME2 BM5,...,ME4 BM1,ME4 BM2,ME4 BM3,ME4 BM4,ME4 BM5,BIG LoBM,ME5 BM2,ME5 BM3,ME5 BM4,BIG HiBM
Date,,,,,,,,,,,,,,,,,,,,,
1963-07-31 23:59:59.999999,1.1287,-0.3632,0.7223,-0.0413,-1.2447,-1.8076,0.1929,-1.0149,-1.9749,-1.1880,...,-0.9115,-1.7733,-1.9168,-1.5745,-1.8574,0.1391,0.4839,1.1360,-0.4285,-1.1045
1963-08-31 23:59:59.999999,4.2396,1.3730,1.4917,2.5068,4.6644,5.5703,4.5220,4.4450,4.4662,8.2451,...,5.5754,4.7469,6.2516,7.6941,5.3456,5.7823,4.2633,4.6341,8.1704,6.3984
1963-09-30 23:59:59.999999,-1.7343,0.6204,-1.0007,-1.5215,-0.3584,-4.0525,-1.5072,-0.8638,-1.4935,-2.9243,...,-2.6644,-2.1463,-1.7882,-3.9641,-2.0002,-1.3752,-0.8081,-0.8497,-0.1912,-3.5033
1963-10-31 23:59:59.999999,0.3778,-0.7329,1.3066,0.1904,2.3711,1.1926,4.2411,2.3526,2.3058,3.9314,...,-0.2415,0.6990,2.5214,4.8524,0.6138,5.3261,1.7420,-0.3354,2.4176,0.4702
1963-11-30 23:59:59.999999,-3.3319,-3.8436,-1.7893,-1.0535,-1.1077,-4.2596,-1.7484,-0.7845,-0.0554,-0.1150,...,-0.9083,-0.6311,-0.7516,1.3596,3.5407,-1.2669,1.0080,-1.6914,-2.1316,1.3496


### Sanity checks

In [6]:
# 1) Indexes align on the same monthly dates
assert factors.index.equals(portfolios.index), "date indexes do not match!"

# 2) No missing values (French uses -99.99 for missing; flag if present)
print("factors  has -99.99 sentinels:", (factors == -99.99).any().any())
print("ports    has -99.99 sentinels:", (portfolios == -99.99).any().any())
print("factors  NaNs:", int(factors.isna().sum().sum()))
print("ports    NaNs:", int(portfolios.isna().sum().sum()))

# 3) Date range
print("range:", factors.index.min().date(), "->", factors.index.max().date(),
      f"({len(factors)} months)")

factors  has -99.99 sentinels: False
ports    has -99.99 sentinels: False
factors  NaNs: 0
ports    NaNs: 0
range: 1963-07-31 -> 2024-12-31 (738 months)


**Quick read of the factors** (means are % per month — annualize by ×12):

- `Mkt-RF` ≈ the equity risk premium
- `RF` is the risk-free rate; everything else is already a *long-short factor return*
  (zero-cost portfolio), so those don't need RF subtracted — only the **test portfolios** do.

That's exactly Step 2.

---

### ✅ Step 1 done

We have aligned monthly factors and 25 test portfolios, sanity-checked.

**Next (Step 2):** build *excess* returns for the 25 portfolios by subtracting `RF`,
and split the factor columns into the three model specifications
(`CAPM`, `FF3`, `FF5`). Tell me when you want to continue and I'll add it.